# Face Match — train, match, analyze

Run the cells **top to bottom**. Every step prints the **same result card**, numbered, one after the other:

| Step | What it does |
|---|---|
| 1 · Analyze | Photo 1 → age, gender, race, emotion |
| 2 · Train | Saves 3 photos of a person under a name (photo 1 + 2 more) |
| 3 · Match? | Photo 2 → is it a trained person? **MATCH / NO MATCH / UNSURE** |
| 3 · Same person? | Photo 1 vs photo 2 (what the old notebook did) |
| 4 · History | Every result so far, in order |
| 5 · Simple UI | All of the above with buttons and a live camera |

**What changed from the old notebook (better matching):**
- `ArcFace` **+** `Facenet512` (instead of `VGG-Face` alone) — a MATCH needs **both** models to agree, otherwise it says **UNSURE** instead of guessing.
- `retinaface` detector with eye alignment (instead of `opencv`) — far fewer missed / badly cropped faces.
- **Training = enrollment**: 3+ photos per person are averaged into one face template, so one bad photo cannot decide the result.
- Quality check: blurry, tiny, or missing faces are rejected with a clear message instead of a wrong answer.
- Each face is embedded together with its mirror image → steadier scores from photo to photo.
- Trained people are saved in `face_gallery.json` and reloaded next time.
- Installs DeepFace from PyPI — the old `git clone` + `pip install -e` is what caused `cannot import name 'DeepFace'`.

> No neural network is re-trained here: the models are already trained on millions of faces. "Training" a person means saving their face embeddings — this is how real face-recognition systems work, and it needs only a few photos.

## 0 · Install (run once)

In [ ]:
# Colab already has TensorFlow. If Colab asks you to restart the session after this, do it and continue below.
!pip install -q "deepface>=0.0.93" tf-keras retina-face ipywidgets

## 0 · Setup — models, settings, matching logic
Change the settings at the top if you need to. `STRICTNESS = 0.9` gives fewer false matches; `1.1` is more forgiving.

In [ ]:
import base64
import html
import json
import os
import time
from datetime import datetime

import cv2
import numpy as np
from IPython.display import HTML, display
from deepface import DeepFace

try:
    from deepface.modules.verification import find_confidence, find_threshold
except ImportError:  # older DeepFace releases
    find_confidence = None
    _THRESHOLDS = {"ArcFace": 0.68, "Facenet512": 0.30, "Facenet": 0.40, "VGG-Face": 0.68, "SFace": 0.593}

    def find_threshold(model_name, distance_metric="cosine"):
        return _THRESHOLDS[model_name]

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ------------------------------------------------------------------ settings
MODELS = ["ArcFace", "Facenet512"]   # two strong models; BOTH must agree for a MATCH
DETECTOR = "retinaface"              # most accurate DeepFace detector, aligns the eyes
METRIC = "cosine"
STRICTNESS = 1.0                     # 0.9 = stricter (fewer false matches), 1.1 = looser
ENROLL_SHOTS = 3                     # photos per person when training
MIN_FACE_PX = 80                     # reject faces smaller than this (too far away)
MIN_DETECTOR_CONF = 0.90             # reject blurry / half-hidden faces
ANALYZE_ACTIONS = ["age", "gender", "race", "emotion"]
GALLERY_FILE = "face_gallery.json"   # trained people are saved here

THRESHOLDS = {m: round(find_threshold(m, METRIC) * STRICTNESS, 4) for m in MODELS}
HISTORY = []                         # every result, in the order it happened


# ------------------------------------------------------------------ images
def to_bgr(src):
    """File path, raw bytes, data-URL or BGR array -> BGR array."""
    if isinstance(src, np.ndarray):
        return src
    if isinstance(src, str) and src.startswith("data:image"):
        src = base64.b64decode(src.split(",", 1)[1])
    if isinstance(src, (bytes, bytearray, memoryview)):
        img = cv2.imdecode(np.frombuffer(bytes(src), np.uint8), cv2.IMREAD_COLOR)
    else:
        img = cv2.imread(str(src))
    if img is None:
        raise ValueError(f"Could not read the image: {str(src)[:60]}")
    return img


def to_data_url(img, boxes=(), color=(40, 180, 60), max_w=320):
    """BGR array (+ face boxes) -> small JPEG data-URL for the result cards."""
    out = img.copy()
    for b in boxes:
        cv2.rectangle(out, (b["x"], b["y"]), (b["x"] + b["w"], b["y"] + b["h"]),
                      color, max(2, out.shape[1] // 160))
    scale = max_w / out.shape[1]
    if scale < 1:
        out = cv2.resize(out, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    _, buf = cv2.imencode(".jpg", out, [cv2.IMWRITE_JPEG_QUALITY, 85])
    return "data:image/jpeg;base64," + base64.b64encode(buf).decode()


# ------------------------------------------------------------------ face + embeddings
def find_face(img):
    """Detect and align the largest face, and check it is good enough to use."""
    faces = DeepFace.extract_faces(img_path=img, detector_backend=DETECTOR, align=True,
                                   enforce_detection=False, color_face="bgr", normalize_face=False)
    faces = [f for f in faces if f.get("confidence", 0) > 0]
    if not faces:
        return {"crop": None, "box": None, "conf": 0.0, "count": 0,
                "problem": "No face found. Look at the camera in good light."}
    best = max(faces, key=lambda f: f["facial_area"]["w"] * f["facial_area"]["h"])
    box = {k: int(best["facial_area"][k]) for k in ("x", "y", "w", "h")}
    conf = float(best["confidence"])
    size = min(box["w"], box["h"])
    problem = None
    if conf < MIN_DETECTOR_CONF:
        problem = f"Face is not clear (detector {conf:.2f} < {MIN_DETECTOR_CONF})."
    elif size < MIN_FACE_PX:
        problem = f"Face is too small ({size}px < {MIN_FACE_PX}px). Move closer."
    crop = np.clip(best["face"], 0, 255).astype(np.uint8)
    return {"crop": crop, "box": box, "conf": conf, "count": len(faces), "size": size, "problem": problem}


def _unit(v):
    v = np.asarray(v, dtype=np.float32)
    return v / (np.linalg.norm(v) + 1e-10)


def embed(crop):
    """One unit vector per model. The face and its mirror image are averaged, which is more stable."""
    vectors = {}
    for m in MODELS:
        pair = [DeepFace.represent(img_path=x, model_name=m, detector_backend="skip",
                                   enforce_detection=False, align=False)[0]["embedding"]
                for x in (crop, cv2.flip(crop, 1))]
        vectors[m] = _unit(_unit(pair[0]) + _unit(pair[1]))
    return vectors


def _confidence(distance, model, same):
    if find_confidence is not None:
        try:
            return float(find_confidence(distance, model, same, METRIC))
        except Exception:
            pass
    t = THRESHOLDS[model]
    return 50 + 50 * (1 - distance / t) if same else 50 * max(0.0, 1 - (distance - t) / t)


def compare(a, b):
    """Compare two {model: vector} sets. Returns per-model rows and MATCH / UNSURE / NO MATCH."""
    rows = []
    for m in MODELS:
        d = float(1 - np.dot(_unit(a[m]), _unit(b[m])))
        ok = d <= THRESHOLDS[m]
        rows.append({"model": m, "distance": round(d, 4), "threshold": THRESHOLDS[m],
                     "same": ok, "confidence": round(_confidence(d, m, ok), 1)})
    votes = sum(r["same"] for r in rows)
    status = "MATCH" if votes == len(rows) else "NO MATCH" if votes == 0 else "UNSURE"
    score = float(np.mean([r["distance"] / r["threshold"] for r in rows]))  # < 1 means same person
    return {"status": status, "rows": rows, "score": score,
            "confidence": round(float(np.mean([r["confidence"] for r in rows])), 1)}


# ------------------------------------------------------------------ trained people (gallery)
def load_gallery():
    if not os.path.exists(GALLERY_FILE):
        return {}
    with open(GALLERY_FILE, encoding="utf-8") as f:
        data = json.load(f)
    usable = {n: p for n, p in data.items() if all(m in p["emb"] for m in MODELS)}
    for n in set(data) - set(usable):
        print(f"'{n}' was trained with other models - train again to use them.")
    return usable


def save_gallery():
    with open(GALLERY_FILE, "w", encoding="utf-8") as f:
        json.dump(GALLERY, f)


def centroid(name):
    return {m: _unit(np.mean(GALLERY[name]["emb"][m], axis=0)) for m in MODELS}


def forget(name=None):
    """forget('Name') removes one person, forget() removes everyone."""
    if name is None:
        GALLERY.clear()
    else:
        GALLERY.pop(name, None)
    save_gallery()
    print("Trained people:", list(GALLERY) or "none")


GALLERY = load_gallery()


# ------------------------------------------------------------------ the four steps
def _result(step, status, headline, **extra):
    res = {"step": step, "status": status, "headline": headline, "time": datetime.now().strftime("%H:%M:%S"),
           "facts": [], "table": [], "images": [], **extra}
    return res


def _finish(res, t0):
    res["seconds"] = round(time.time() - t0, 2)
    HISTORY.append(res)
    res["n"] = len(HISTORY)
    return res


def _quality_fact(face):
    return ("Face quality", f"detector {face['conf']:.2f}, size {face.get('size', 0)}px"
            + (f", {face['count']} faces (largest used)" if face["count"] > 1 else "") + " ✓")


def run_analyze(src):
    """Age, gender, race and emotion for every face in the photo."""
    t0 = time.time()
    try:
        img = to_bgr(src)
        try:
            faces = DeepFace.analyze(img_path=img, actions=ANALYZE_ACTIONS, detector_backend=DETECTOR,
                                     enforce_detection=True, silent=True)
        except ValueError:
            res = _result("ANALYZE", "NO FACE", "No face found. Look at the camera in good light.")
            res["images"].append(("Photo", to_data_url(img)))
            return _finish(res, t0)
        res = _result("ANALYZE", "OK", f"{len(faces)} face(s) found")
        for i, f in enumerate(faces, 1):
            r = f["region"]
            row = {"face": i, "position": f"x={r['x']} y={r['y']} w={r['w']} h={r['h']}"}
            if "age" in f:
                row["age"] = int(f["age"])
            if "dominant_gender" in f:
                row["gender"] = f"{f['dominant_gender']} ({float(f['gender'][f['dominant_gender']]):.1f}%)"
            if "dominant_race" in f:
                row["race"] = f"{f['dominant_race']} ({float(f['race'][f['dominant_race']]):.1f}%)"
            if "dominant_emotion" in f:
                row["emotion"] = f"{f['dominant_emotion']} ({float(f['emotion'][f['dominant_emotion']]):.1f}%)"
            res["table"].append(row)
        boxes = [{k: int(f["region"][k]) for k in ("x", "y", "w", "h")} for f in faces]
        res["images"].append(("Detected faces", to_data_url(img, boxes)))
    except Exception as e:
        res = _result("ANALYZE", "ERROR", str(e))
    return _finish(res, t0)


def run_enroll(name, sources):
    """'Train' a person: store face embeddings from a few photos under a name."""
    t0 = time.time()
    name = (name or "").strip()
    if not name:
        return _finish(_result("TRAIN", "ERROR", "Type a name first."), t0)
    try:
        person = GALLERY.get(name)
        ref = centroid(name) if person else None
        added, notes, images = 0, [], []
        for i, src in enumerate(sources, 1):
            img = to_bgr(src)
            face = find_face(img)
            if face["problem"]:
                notes.append((f"Photo {i}", "✗ skipped - " + face["problem"]))
                images.append((f"Photo {i} ✗", to_data_url(img, [face["box"]] if face["box"] else [], (40, 40, 220))))
                continue
            vec = embed(face["crop"])
            if ref is not None and compare(vec, ref)["status"] == "NO MATCH":
                notes.append((f"Photo {i}", f"✗ skipped - does not look like the other photos of {name}"))
                images.append((f"Photo {i} ✗", to_data_url(img, [face["box"]], (40, 40, 220))))
                continue
            if person is None:
                person = GALLERY[name] = {"emb": {m: [] for m in MODELS}, "thumb": None}
            for m in MODELS:
                person["emb"][m].append(vec[m].round(6).tolist())
            if person["thumb"] is None:
                x, y, w, h = (face["box"][k] for k in ("x", "y", "w", "h"))
                person["thumb"] = to_data_url(img[max(0, y - h // 4):y + h + h // 4, max(0, x - w // 4):x + w + w // 4], max_w=160)
            ref = centroid(name)
            added += 1
            notes.append((f"Photo {i}", "✓ added  (" + _quality_fact(face)[1] + ")"))
            images.append((f"Photo {i} ✓", to_data_url(img, [face["box"]])))
        if added:
            save_gallery()
            for other in GALLERY:
                if other != name and compare(ref, centroid(other))["status"] == "MATCH":
                    notes.append(("Warning", f"{name} looks the same as '{other}' who is already trained"))
        total = len(GALLERY[name]["emb"][MODELS[0]]) if name in GALLERY else 0
        status = "OK" if added else "ERROR"
        res = _result("TRAIN", status, f"{name}: {added} of {len(sources)} photo(s) added - {total} in total")
        res["facts"] = notes + [("Trained people", ", ".join(f"{n} ({len(p['emb'][MODELS[0]])})" for n, p in GALLERY.items()) or "none")]
        res["images"] = images
    except Exception as e:
        res = _result("TRAIN", "ERROR", str(e))
    return _finish(res, t0)


def run_identify(src):
    """Who is this? Compare a new photo with every trained person."""
    t0 = time.time()
    try:
        img = to_bgr(src)
        face = find_face(img)
        if face["problem"]:
            res = _result("MATCH?", "NO FACE", face["problem"])
            res["images"].append(("Photo", to_data_url(img, [face["box"]] if face["box"] else [], (40, 40, 220))))
            return _finish(res, t0)
        if not GALLERY:
            return _finish(_result("MATCH?", "ERROR", "Nobody is trained yet - run the TRAIN step first."), t0)
        vec = embed(face["crop"])
        ranking = sorted(((n, compare(vec, centroid(n))) for n in GALLERY), key=lambda x: x[1]["score"])
        best, cmp = ranking[0]
        headline = {"MATCH": f"This is {best}",
                    "UNSURE": f"Maybe {best} - the two models disagree, take another photo",
                    "NO MATCH": "Unknown person - does not match anyone trained"}[cmp["status"]]
        res = _result("MATCH?", cmp["status"], headline, confidence=cmp["confidence"])
        res["facts"] = [("Closest person", best), ("Match confidence", f"{cmp['confidence']}%"), _quality_fact(face)]
        res["facts"] += [(f"#{i} {n}", f"score {c['score']:.2f}  ({c['status']})") for i, (n, c) in enumerate(ranking[:3], 1)]
        res["table"] = cmp["rows"]
        color = (40, 180, 60) if cmp["status"] == "MATCH" else (40, 40, 220) if cmp["status"] == "NO MATCH" else (0, 170, 230)
        res["images"] = [("New photo", to_data_url(img, [face["box"]], color)), (f"Trained: {best}", GALLERY[best]["thumb"])]
    except Exception as e:
        res = _result("MATCH?", "ERROR", str(e))
    return _finish(res, t0)


def run_verify(src1, src2):
    """Are the people in two photos the same person?"""
    t0 = time.time()
    try:
        imgs = [to_bgr(src1), to_bgr(src2)]
        faces = [find_face(i) for i in imgs]
        bad = [f"Photo {i}: {f['problem']}" for i, f in enumerate(faces, 1) if f["problem"]]
        if bad:
            res = _result("SAME PERSON?", "NO FACE", " / ".join(bad))
        else:
            cmp = compare(embed(faces[0]["crop"]), embed(faces[1]["crop"]))
            headline = {"MATCH": "Same person", "UNSURE": "Not sure - the two models disagree",
                        "NO MATCH": "Different persons"}[cmp["status"]]
            res = _result("SAME PERSON?", cmp["status"], headline, confidence=cmp["confidence"])
            res["facts"] = [("Match confidence", f"{cmp['confidence']}%"), ("Photo 1 quality", _quality_fact(faces[0])[1]),
                            ("Photo 2 quality", _quality_fact(faces[1])[1])]
            res["table"] = cmp["rows"]
        res["images"] = [(f"Photo {i}", to_data_url(img, [f["box"]] if f["box"] else []))
                         for i, (img, f) in enumerate(zip(imgs, faces), 1)]
    except Exception as e:
        res = _result("SAME PERSON?", "ERROR", str(e))
    return _finish(res, t0)


# ------------------------------------------------------------------ output (same card everywhere)
COLORS = {"MATCH": "#1f9d55", "NO MATCH": "#d64545", "UNSURE": "#d98c00", "OK": "#2b6cb0",
          "NO FACE": "#d98c00", "ERROR": "#d64545"}
ICONS = {"MATCH": "✅", "NO MATCH": "❌", "UNSURE": "⚠️", "OK": "✔️", "NO FACE": "⚠️", "ERROR": "❌"}


def card(res):
    esc = lambda v: html.escape(str(v))
    color = COLORS.get(res["status"], "#555")
    n = res.get("n", "")
    facts = "".join(f"<tr><td style='padding:2px 12px 2px 0;color:#666'>{esc(k)}</td><td>{esc(v)}</td></tr>"
                    for k, v in res["facts"])
    table = ""
    if res["table"]:
        cols = list(res["table"][0])
        head = "".join(f"<th style='text-align:left;padding:4px 10px;border-bottom:1px solid #ddd'>{esc(c)}</th>" for c in cols)
        body = "".join("<tr>" + "".join(
            f"<td style='padding:4px 10px'>{'✓ yes' if v is True else '✗ no' if v is False else esc(v)}</td>" for v in r.values())
            + "</tr>" for r in res["table"])
        table = f"<table style='border-collapse:collapse;margin-top:8px;font-size:13px'><tr>{head}</tr>{body}</table>"
    images = "".join(f"<figure style='margin:0'><img src='{src}' style='max-height:220px;border-radius:6px;display:block'>"
                     f"<figcaption style='font-size:12px;color:#666;text-align:center'>{esc(cap)}</figcaption></figure>"
                     for cap, src in res["images"] if src)
    return f"""
<div style="font-family:system-ui,Segoe UI,Arial,sans-serif;color:#222;background:#fff;border:1px solid #ddd;
            border-left:8px solid {color};border-radius:8px;padding:12px 16px;margin:10px 0;max-width:900px">
  <div style="font-size:12px;color:#888">#{n} · STEP: {esc(res['step'])} · {res['time']} · {res.get('seconds', '')} s</div>
  <div style="font-size:22px;font-weight:700;color:{color};margin:4px 0">{ICONS.get(res['status'], '')} {esc(res['status'])}
     <span style="font-size:16px;font-weight:500;color:#222"> — {esc(res['headline'])}</span></div>
  <table style="font-size:13px">{facts}</table>{table}
  <div style="display:flex;gap:12px;flex-wrap:wrap;margin-top:10px">{images}</div>
</div>"""


def show(res):
    display(HTML(card(res)))
    return res


def text_line(i, res):
    extra = f"  confidence {res['confidence']}%" if "confidence" in res else ""
    return f"#{i:<3} {res['time']}  {res['step']:<13} {res['status']:<9} {res['headline']}{extra}"


def print_history():
    """Every step so far, one after the other, in the same format."""
    print(f"{len(HISTORY)} result(s)\n" + "-" * 90)
    for i, res in enumerate(HISTORY, 1):
        print(text_line(i, res))


# ------------------------------------------------------------------ warm up (downloads weights once)
print("Loading models (first run downloads the weights)...")
for m in MODELS:
    DeepFace.build_model(m)
DeepFace.extract_faces(np.zeros((160, 160, 3), np.uint8), detector_backend=DETECTOR, enforce_detection=False)
print(f"Ready.  models={MODELS}  detector={DETECTOR}  thresholds={THRESHOLDS}")
print(f"Running in {'Google Colab' if IN_COLAB else 'local Jupyter'}.  Trained people: {list(GALLERY) or 'none'}")

## 0 · Camera

In [ ]:
def capture(label="Capture"):
    """Take one webcam photo and return it as a BGR array.

    Colab: the browser camera opens under this cell - click the button.
    Local Jupyter / VS Code: an OpenCV window opens - press SPACE to take the photo, ESC to cancel.
    """
    if IN_COLAB:
        from google.colab.output import eval_js
        from IPython.display import Javascript
        display(Javascript('''
        async function takePhoto(label) {
          const div = document.createElement('div');
          const btn = document.createElement('button');
          btn.textContent = '📷 ' + label;
          btn.style.cssText = 'font-size:16px;padding:8px 18px;margin:6px 0;cursor:pointer';
          const video = document.createElement('video');
          video.style.cssText = 'display:block;max-width:480px;border-radius:8px;transform:scaleX(-1)';
          const stream = await navigator.mediaDevices.getUserMedia({video: true});
          document.body.appendChild(div);
          div.appendChild(btn);
          div.appendChild(video);
          video.srcObject = stream;
          await video.play();
          google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
          await new Promise((resolve) => btn.onclick = resolve);
          const canvas = document.createElement('canvas');
          canvas.width = video.videoWidth;
          canvas.height = video.videoHeight;
          canvas.getContext('2d').drawImage(video, 0, 0);
          stream.getVideoTracks()[0].stop();
          div.remove();
          return canvas.toDataURL('image/jpeg', 0.92);
        }'''))
        return to_bgr(eval_js(f"takePhoto({json.dumps(label)})"))

    cam = cv2.VideoCapture(0)
    if not cam.isOpened():
        raise RuntimeError("No webcam found - set a file path instead of None.")
    try:
        while True:
            ok, frame = cam.read()
            if not ok:
                raise RuntimeError("Could not read from the webcam.")
            preview = cv2.flip(frame, 1)
            cv2.putText(preview, f"{label}: SPACE = take photo, ESC = cancel", (10, 28),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.imshow("Camera", preview)
            key = cv2.waitKey(1) & 0xFF
            if key == 32:
                return frame
            if key == 27:
                raise RuntimeError("Capture cancelled.")
    finally:
        cam.release()
        cv2.destroyAllWindows()


def photo(path, label):
    """Use the file if a path is given, otherwise take a webcam photo."""
    return to_bgr(path) if path else capture(label)


print("Camera ready.")

## 1 · Analyze — capture photo 1
Leave `PHOTO_1 = None` to use the webcam, or put a file name such as `"img1.jpg"`.

In [ ]:
PHOTO_1 = None

photo1 = photo(PHOTO_1, "Capture photo 1")
show(run_analyze(photo1));

## 2 · Train — save a person
Photo 1 is used as the first training photo; then take 2 more (turn your head **a little** left, then right).
Run this cell again with another name to train more people.

In [ ]:
PERSON_NAME = "Person 1"
TRAIN_FILES = []      # optional: ["a.jpg", "b.jpg", "c.jpg"] instead of the webcam

shots = [to_bgr(p) for p in TRAIN_FILES] or [photo1] + [
    capture(f"Training photo {i} of {ENROLL_SHOTS}") for i in range(2, ENROLL_SHOTS + 1)]
show(run_enroll(PERSON_NAME, shots));

## 3 · Match? — capture photo 2
Is the new photo a trained person? And is it the same person as photo 1?

In [ ]:
PHOTO_2 = None

photo2 = photo(PHOTO_2, "Capture photo 2")
show(run_identify(photo2))
show(run_verify(photo1, photo2));

## 4 · History — every result, one after the other

In [ ]:
print_history()

## 5 · Simple UI — everything with buttons
Type a name and click **Train** 3 times (move your head a little between clicks), then click **Match?**.
Results stack up below the buttons in order. `forget("Name")` removes a person, `forget()` removes everyone.

In [ ]:
def _ui_call(fn, *args):
    try:
        res = fn(*args)
    except Exception as e:  # never leave the UI hanging
        res = _finish(_result("UI", "ERROR", str(e)), time.time())
    return card(res)


UI_HTML = r"""
<div style="font-family:system-ui,Segoe UI,Arial,sans-serif;color:#222;background:#f7f8fa;border:1px solid #ddd;
            border-radius:10px;padding:14px;max-width:940px">
  <div style="font-size:20px;font-weight:700;margin-bottom:8px">Face Match</div>
  <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-start">
    <div>
      <video id="fm-video" autoplay playsinline muted
             style="width:400px;max-width:100%;border-radius:8px;background:#000;transform:scaleX(-1)"></video>
      <div style="font-size:12px;color:#666">or use a file:
        <input id="fm-file" type="file" accept="image/*" style="font-size:12px"></div>
    </div>
    <div style="display:flex;flex-direction:column;gap:8px;min-width:240px">
      <label style="font-size:13px">Person name (for training)<br>
        <input id="fm-name" placeholder="e.g. Rahul" style="width:100%;padding:6px;font-size:14px"></label>
      <button data-act="enroll">1 · Train: add this photo to the person</button>
      <button data-act="identify">2 · Match? Who is this</button>
      <button data-act="verify">3 · Same person as the previous photo?</button>
      <button data-act="analyze">4 · Analyze: age / gender / emotion</button>
      <div id="fm-status" style="font-size:13px;color:#2b6cb0;min-height:18px"></div>
      <div style="font-size:12px;color:#666">Tip: train 3+ photos per person<br>(straight, a little left, a little right).</div>
    </div>
  </div>
  <div id="fm-results"></div>
</div>
<style>#fm-video+div input{color:#222} [data-act]{font-size:14px;padding:8px;cursor:pointer;border-radius:6px;
  border:1px solid #bbb;background:#fff;color:#222;text-align:left} [data-act]:disabled{opacity:.5}</style>
<script>
(async () => {
  const video = document.getElementById('fm-video');
  const status = document.getElementById('fm-status');
  const results = document.getElementById('fm-results');
  let filePhoto = null, previous = null;
  try {
    video.srcObject = await navigator.mediaDevices.getUserMedia({video: true});
  } catch (e) { status.textContent = 'No camera - choose a file instead.'; }
  document.getElementById('fm-file').onchange = (ev) => {
    const f = ev.target.files[0]; if (!f) return;
    const r = new FileReader(); r.onload = () => { filePhoto = r.result; status.textContent = 'Using file: ' + f.name; };
    r.readAsDataURL(f);
  };
  function snap() {
    if (filePhoto) { const p = filePhoto; filePhoto = null; document.getElementById('fm-file').value = ''; return p; }
    if (!video.videoWidth) return null;
    const c = document.createElement('canvas');
    c.width = video.videoWidth; c.height = video.videoHeight;
    c.getContext('2d').drawImage(video, 0, 0);
    return c.toDataURL('image/jpeg', 0.92);
  }
  const resize = () => { if (window.google && google.colab) google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true); };
  document.querySelectorAll('[data-act]').forEach(btn => btn.onclick = async () => {
    const act = btn.dataset.act, img = snap();
    if (!img) { status.textContent = 'No photo - allow the camera or choose a file.'; return; }
    let args = [img];
    if (act === 'enroll') args = [document.getElementById('fm-name').value, img];
    if (act === 'verify') {
      if (!previous) { previous = img; status.textContent = 'Saved as the first photo - click again for the second one.'; return; }
      args = [previous, img];
    }
    document.querySelectorAll('[data-act]').forEach(b => b.disabled = true);
    status.textContent = 'Working...';
    try {
      const r = await google.colab.kernel.invokeFunction('facematch.' + act, args, {});
      results.insertAdjacentHTML('beforeend', r.data['application/json'].html);
      status.textContent = 'Done.';
    } catch (e) { status.textContent = 'Error: ' + e; }
    previous = img;
    document.querySelectorAll('[data-act]').forEach(b => b.disabled = false);
    resize();
  });
  resize();
})();
</script>
"""


def launch_ui():
    if IN_COLAB:
        from google.colab import output
        from IPython.display import JSON
        output.register_callback("facematch.enroll", lambda name, img: JSON({"html": _ui_call(run_enroll, name, [img])}))
        output.register_callback("facematch.identify", lambda img: JSON({"html": _ui_call(run_identify, img)}))
        output.register_callback("facematch.verify", lambda a, b: JSON({"html": _ui_call(run_verify, a, b)}))
        output.register_callback("facematch.analyze", lambda img: JSON({"html": _ui_call(run_analyze, img)}))
        display(HTML(UI_HTML))
        return

    # Local Jupyter / VS Code: same buttons, photo from a file or from the OpenCV webcam window.
    import ipywidgets as w
    name = w.Text(placeholder="e.g. Rahul", description="Name")
    upload = w.FileUpload(accept="image/*", multiple=False, description="Choose photo")
    use_cam = w.Checkbox(value=True, description="Use webcam when no file is chosen")
    out = w.Output()
    state = {"previous": None}

    def take():
        files = upload.value
        if files:
            item = files[0] if isinstance(files, (list, tuple)) else next(iter(files.values()))
            try:
                upload.value = ()
            except Exception:  # ipywidgets 7
                upload.value.clear()
            return to_bgr(item["content"])
        if use_cam.value:
            return capture("Photo")
        raise RuntimeError("Choose a photo first.")

    def action(kind):
        def go(_):
            with out:
                try:
                    img = take()
                    if kind == "verify" and state["previous"] is None:
                        state["previous"] = img
                        print("Saved as the first photo - click again for the second one.")
                        return
                    html_card = {"enroll": lambda: _ui_call(run_enroll, name.value, [img]),
                                 "identify": lambda: _ui_call(run_identify, img),
                                 "verify": lambda: _ui_call(run_verify, state["previous"], img),
                                 "analyze": lambda: _ui_call(run_analyze, img)}[kind]()
                    state["previous"] = img
                    display(HTML(html_card))
                except Exception as e:
                    print("Error:", e)
        return go

    buttons = []
    for kind, text in [("enroll", "1 · Train"), ("identify", "2 · Match?"),
                       ("verify", "3 · Same person?"), ("analyze", "4 · Analyze")]:
        b = w.Button(description=text, layout=w.Layout(width="160px"))
        b.on_click(action(kind))
        buttons.append(b)
    display(w.VBox([w.HTML("<b style='font-size:18px'>Face Match</b>"), w.HBox([name, upload]), use_cam,
                    w.HBox(buttons), out]))


launch_ui()